In [ ]:
import brainpy as bp
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import jax
import jax.numpy as jnp
import sys
import os

src_dir = os.path.abspath(os.path.join('../../../'))
sys.path.insert(0, src_dir)
import src
# plt.style.use('../../../foresight.mplstyle')


: 

In [ ]:
model = src.models.FNS
fixed_params = {
    # 'N_e': 1000,
    'J_e': 0.0008,
    'n_ext': 100,
    'omega_ee': 0.036,
    'omega_ei': 0.0564,
    'omega_ie': 0.0444,
    'omega_ii': 0.078,
}
monitors = ['E.spike', 'E.V', 'E.input']
stats = {"rate": src.stats.firing_rate,
         "susceptibility": src.stats.susceptibility(10),
         "spike_spectrum": src.stats.spike_spectrum(10),
         "temporal_average": src.stats.temporal_average,
         "grand_distribution": src.stats.grand_distribution(1000),
         }
duration = 1000.0
transient = 200.0

run = src.stats.create_run(model, fixed_params, monitors, duration, transient)
stats_run = src.stats.create_stats_run(run, stats)

params = {
    'nu': np.array([8.0, 10.0, 8.0, 10.0]),
    'delta': np.array([4.0, 4.0, 8.0, 10.0]),
    'key': np.array(jax.random.split(jax.random.PRNGKey(42), 4))
    # 'key': jnp.repeat(jax.random.PRNGKey(42)[None, :], 2, axis=0)
}

static_params = {'N_e': np.array([1000, 2000, 3000, 4000])}

# result = jax.vmap(run)(params)
# result = jax.vmap(stats_run)(params)

In [ ]:
# result = jax.vmap(stats_run)(params)
result = src.stats.progress_vmap(stats_run, batch_size=1)(params, frozenset(static_params.items())) # * The frozenset is important so that the static params are hashable

In [ ]:
# * Plot a distribution
h = result['grand_distribution']['E.input'][0][0]
bs = result['grand_distribution']['E.input'][1][0]
h = h / jnp.sum(h)  # normalize the histogram

# * Plot the distribution
plt.figure(figsize=(6, 4))
plt.semilogy(bs[20:], h[20:])
plt.xlabel('input')

In [ ]:
# * Plot the spike spectrum

fig = plt.figure(figsize=(5, 3))
ax = fig.add_subplot(111)
spectrum = result['spike_spectrum']['E.spike'][0]
freqs = jnp.fft.fftfreq(spectrum.shape[0], bp.share["dt"]/1000)
idxs = freqs > 0
ax.plot(freqs[idxs], spectrum[idxs], lw=2)
ax.set_xlim(0, 100)
